# Understanding Mathematical Concepts in Autograd Operations

*First, watch the video in the Solution section.*

This task focuses on implementing basic automatic differentiation (autograd) mechanisms for neural networks. The operations of addition, multiplication, and ReLU are fundamental to neural network computations and their training through backpropagation.

---

## Mathematical Foundations


### Addition (`__add__`)
* **Forward Pass:** For two scalar values $a$ and $b$, their sum $s$ is:
  $$s = a + b$$
* **Backward Pass:** The derivative of $s$ with respect to both $a$ and $b$ is $1$. During backpropagation, the gradient of the output is passed directly to both inputs without modification.

### Multiplication (`__mul__`)
* **Forward Pass:** For two scalar values $a$ and $b$, their product $p$ is:
  $$p = a \times b$$
* **Backward Pass:** The gradient of $p$ with respect to $a$ is $b$, and with respect to $b$ is $a$. During backpropagation, each input's gradient is the product of the *other* input and the output's gradient (chain rule).

### ReLU Activation (`relu`)


* **Forward Pass:** The Rectified Linear Unit (ReLU) function is defined as:
  $$R(x) = \max(0, x)$$
  This function outputs $x$ if $x$ is positive, and $0$ otherwise.
* **Backward Pass:** The derivative of the ReLU function is $1$ for $x > 0$ and $0$ for $x \le 0$. The gradient is propagated through the function only if the input is positive; otherwise, it stops (acting as a gate).

---

## Conceptual Application in Neural Networks

* **Addition and Multiplication:** These operations are ubiquitous in neural networks, forming the mathematical basis for computing the weighted sums of inputs within individual neurons ($z = \sum w_i x_i + b$).
* **ReLU Activation:** Commonly used as an activation function in neural networks due to its simplicity and effectiveness in introducing non-linearity. This non-linearity makes learning complex, high-dimensional patterns possible.

## Why This Matters
Understanding these operations and their implications on gradient flow is crucial for designing and training effective neural network models. By implementing these from scratch, you gain deeper insights into the under-the-hood workings of more sophisticated deep learning libraries like PyTorch or TensorFlow.

[Problem Desc](#https://www.deep-ml.com/problems/26)

In [7]:
class Value:
	def __init__(self, data, _children=(), _op=''):
		self.data = data
		self.grad = 0
		self._backward = lambda: None
		self._prev = set(_children)
		self._op = _op
	def __repr__(self):
		grad_print = int(self.grad) if self.grad == int(self.grad) else self.grad
		return f"Value(data={self.data}, grad={grad_print})"

	def __add__(self, other):
		 # Implement addition here
		other = other if isinstance(other, Value) else Value(other)

		out = Value(self.data + other.data, _children=(self, other), _op='+')

		def _backward():
			self.grad += 1.0 * out.grad
			other.grad += 1.0 * out.grad

		out._backward = _backward
		return out

	def __mul__(self, other):
		# Implement multiplication here
		other = other if isinstance(other, Value) else Value(other)

		out = Value(self.data * other.data, _children=(self, other), _op='*')

		def _backward():
			self.grad += other.data * out.grad
			other.grad += self.data * out.grad

		out._backward = _backward
		return out

	def relu(self):
		# Implement ReLU here
		out_data = self.data if self.data > 0 else 0
		out = Value(out_data, _children=(self,), _op='ReLU')

		def _backward():
			self.grad += (out.data > 0) * out.grad
   
		out._backward = _backward
		return out
  
	def backward(self):
		# Implement backward pass here
		topo = []
		visited = set()

		def build_topo(v):
			if v not in visited:
				visited.add(v)

				for child in v._prev:
					build_topo(child)
				topo.append(v)

		build_topo(self)

		self.grad = 1.0

		for node in reversed(topo):
			node._backward()

In [8]:
# Test Case 1
a = Value(2);b = Value(3);c = Value(10);d = a + b * c  ;e = Value(7) * Value(2);f = e + d;g = f.relu()  
g.backward()
print(a,b,c,d,e,f,g)


# Expected Output:
#  Value(data=2, grad=1) Value(data=3, grad=10) Value(data=10, grad=3) Value(data=32, grad=1) Value(data=14, grad=1) Value(data=46, grad=1) Value(data=46, grad=1)

Value(data=2, grad=1) Value(data=3, grad=10) Value(data=10, grad=3) Value(data=32, grad=1) Value(data=14, grad=1) Value(data=46, grad=1) Value(data=46, grad=1)
